In [0]:
%sql
INSERT INTO smart_meters.gold.fact_consumption
SELECT
    fc.energy_spent,
    fc.money_spent,
    fc.dim_tariff_sk,
    fc.dim_household_lclid,
    fc.time_id as dim_time_id,
    t.sk_weather as weather_sk
FROM smart_meters.silver.fact_consumption fc
LEFT JOIN smart_meters.gold.dim_household hh
    ON fc.dim_household_lclid = hh.lclid
LEFT JOIN smart_meters.silver.dim_time t
    ON fc.time_id = t.time_id
LEFT JOIN smart_meters.gold.dim_weather w
    ON t.sk_weather = w.sk_weather
LEFT JOIN smart_meters.gold.dim_tariffs tar
    ON fc.dim_tariff_sk = tar.sk
WHERE fc.time_id NOT IN (SELECT DISTINCT dim_time_id FROM smart_meters.gold.fact_consumption)

In [0]:
%sql
-- O MERGE compara os IDs de tempo (ex: 2013010118)
MERGE INTO smart_meters.gold.dim_time AS target
USING smart_meters.silver.dim_time AS source
ON target.id = source.time_id

-- Como o tempo não muda, não temos o bloco 'WHEN MATCHED'
-- Se a data/hora for nova (NOT MATCHED), inserimos:
WHEN NOT MATCHED THEN
  INSERT (id, year, month, day, hour, season, holiday_type)
  VALUES (
      source.time_id, 
      source.year, 
      source.month, 
      source.day, 
      source.hour, 
      source.season, 
      source.holiday_type
  );

In [0]:
%sql
-- 1. Identificar mudanças (comparar Silver com Gold)
MERGE INTO smart_meters.gold.dim_tariffs AS target
USING smart_meters.silver.dim_tariffs AS source
ON target.sk = source.sk AND target.is_current = true
WHEN MATCHED AND target.price <> source.price THEN
  -- Fecha a tarifa antiga
  UPDATE SET target.valid_to = CURRENT_DATE(), target.is_current = false
WHEN NOT MATCHED THEN
  -- Insere a nova
  INSERT (sk, id, type, period, price, valid_from, is_current)
  VALUES (source.sk, source.tariff_id, source.type, source.period, source.price, CURRENT_DATE(), true);